# Credit Card Segmentation: Production Pipeline & Cluster Profiling

Goal: Execute a clean data processing pipeline, compare business profiles for k=5, k=6, and k=7, and export the final trained assets for the interactive Streamlit dashboard.

Key Steps:
1. Load & Clean Data
2. Scale Features
3. Profile Candidate Clusters (k=5, 6, 7)
4. Train Final Selected Model
5. Export Model & Scaler Assets

In [1]:
import pandas as pd
import numpy as np
import joblib 

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [2]:
# --- STEP 1: LOAD & CLEAN ---
df_raw = pd.read_csv('C:\\customer_segmentation\\Credit-Card-Customer-Segmentation\\data\\CC GENERAL.csv')
df = df_raw.drop('CUST_ID', axis=1)

# Explicit Imputation (Avoiding Copy-on-Write warnings)
df['MINIMUM_PAYMENTS'] = df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].median())
df['CREDIT_LIMIT'] = df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].median())

# --- STEP 2: SCALE ---
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)
df_scaled = pd.DataFrame(df_scaled, columns=df.columns)

print("Pipeline Stage 1 & 2 Complete. Data shape:", df_scaled.shape)

Pipeline Stage 1 & 2 Complete. Data shape: (8950, 17)


In [3]:
# The candidates we identified from the Silhouette/Elbow charts
candidate_ks = [5, 6, 7]

# Features that are easiest for business stakeholders to understand
business_features = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT', 'PAYMENTS', 'PRC_FULL_PAYMENT', 'PURCHASES_FREQUENCY']

for k in candidate_ks:
    print(f"\n{'='*40}")
    print(f"BUSINESS PROFILE FOR k = {k}")
    print(f"{'='*40}")
    
    # Train temporary model
    temp_kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    temp_labels = temp_kmeans.fit_predict(df_scaled)
    
    # Add labels temporarily to our UN-SCALED data (so we see real dollar amounts)
    df_temp = df.copy()
    df_temp['Cluster'] = temp_labels
    
    # Calculate means and cluster sizes
    profile = df_temp.groupby('Cluster')[business_features].mean().round(0)
    profile['Customer_Count'] = df_temp['Cluster'].value_counts()
    
    # Display the table with a background gradient so high values pop out in color
    display(profile.style.background_gradient(cmap='Blues'))


BUSINESS PROFILE FOR k = 5


,BALANCE,PURCHASES,CASH_ADVANCE,CREDIT_LIMIT,PAYMENTS,PRC_FULL_PAYMENT,PURCHASES_FREQUENCY,Customer_Count
Cluster,,,,,,,,
0,4903.000000,553.000000,4983.000000,8063.000000,3859.000000,0.000000,0.000000,987
1,3589.000000,7816.000000,662.000000,9770.000000,7409.000000,0.000000,1.000000,395
2,930.000000,1300.000000,227.000000,4272.000000,1389.000000,0.000000,1.000000,3164
3,1526.000000,255.000000,795.000000,3244.000000,959.000000,0.000000,0.000000,3047
4,111.000000,335.000000,326.000000,3687.000000,1077.000000,0.000000,0.000000,1357



BUSINESS PROFILE FOR k = 6


,BALANCE,PURCHASES,CASH_ADVANCE,CREDIT_LIMIT,PAYMENTS,PRC_FULL_PAYMENT,PURCHASES_FREQUENCY,Customer_Count
Cluster,,,,,,,,
0,4869.000000,551.000000,5073.000000,8057.000000,3941.000000,0.000000,0.000000,958
1,142.000000,1408.000000,49.000000,4856.000000,1498.000000,1.000000,1.000000,1070
2,1482.000000,1427.000000,351.000000,4354.000000,1528.000000,0.000000,1.000000,2442
3,1500.000000,234.000000,808.000000,3206.000000,946.000000,0.000000,0.000000,2959
4,115.000000,310.000000,342.000000,3634.000000,1058.000000,0.000000,0.000000,1284
5,4133.000000,9992.000000,690.000000,10646.000000,9465.000000,0.000000,1.000000,237



BUSINESS PROFILE FOR k = 7


,BALANCE,PURCHASES,CASH_ADVANCE,CREDIT_LIMIT,PAYMENTS,PRC_FULL_PAYMENT,PURCHASES_FREQUENCY,Customer_Count
Cluster,,,,,,,,
0,4820.000000,519.000000,4912.000000,7956.000000,3874.000000,0.000000,0.000000,991
1,128.000000,1292.000000,58.000000,5018.000000,1541.000000,1.000000,1.000000,1113
2,1427.000000,1439.000000,322.000000,4426.000000,1555.000000,0.000000,1.000000,2417
3,3931.000000,779.000000,890.000000,4182.000000,1380.000000,0.000000,0.000000,55
4,836.000000,391.000000,1066.000000,2444.000000,602.000000,0.000000,0.000000,659
5,4106.000000,10002.000000,693.000000,10657.000000,9494.000000,0.000000,1.000000,236
6,1121.000000,255.000000,614.000000,3364.000000,1002.000000,0.000000,0.000000,3479


In [4]:
# --- STEP 3: TRAIN FINAL MODEL ---
# CHANGE THIS NUMBER to your final decision based on the tables above
FINAL_K = 5 

print(f"Training final production model with k={FINAL_K}...")
final_kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init='auto')
final_labels = final_kmeans.fit_predict(df_scaled)

# Add final labels to data
df_final = df.copy()
df_final['Cluster'] = final_labels

# --- STEP 4: EXPORT ASSETS FOR STREAMLIT ---
# We save the model and the scaler. 
# We MUST save the scaler so the Streamlit app knows how to process new user inputs!
joblib.dump(final_kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'data_scaler.pkl')
df_final.to_csv('segmented_customers.csv', index=False)

print("Success! 'kmeans_model.pkl', 'data_scaler.pkl', and 'segmented_customers.csv' saved to disk.")

Training final production model with k=5...
Success! 'kmeans_model.pkl', 'data_scaler.pkl', and 'segmented_customers.csv' saved to disk.
